- Extracting 5 questions and their answers from a chunk (e.g. BART QA generator model)
- Extracting nodes and relations from these QA pairs (with or without an agent)

In [ ]:
import os
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.documents import Document
from langchain_experimental.graph_transformers import LLMGraphTransformer
import json

LLM_MODEL_NAME="jaro-jm/llm:latest"
# LLM_MODEL_NAME="SpeakLeash/bielik-11b-v2.3-instruct:Q8_0"
os.environ["HSA_OVERRIDE_GFX_VERSION"]="11.0.0"

llm_text = ChatOllama(model=LLM_MODEL_NAME, temperature=0, reasoning=False, num_ctx=25000, num_predict=15000)

text = """Donald Trump od początku traktował konflikt w Ukrainie jako nieswoją wojnę. Nie miał cierpliwości ani zrozumienia dla ukraińskich racji. Błędnie zakładał też, że zdoła oderwać Rosję od sojuszu z Chinami. Tymczasem oba kraje mocno zacieśniły współpracę gospodarczą, a Chiny dostarczają dziś Rosjanom maszyny i technologie, na które monopol miał kiedyś Zachód. Moskwa i Pekin pamiętają czasy, gdy były podzielone. Putin i Xi są zgodni, że rozpad Związku Radzieckiego nie był koniecznością, tylko skutkiem błędów Kremla. Rosję i Chiny dzieli przepaść kulturalna, bo Rosjanie nie chcą być Chińczykami, ale to akurat ułatwia relacje. Współpraca z Europą grozi w oczach Putina zarażeniem rosyjskiego społeczeństwa „demokratycznym wirusem”, a w kontaktach z Chinami takiego ryzyka nie ma. Rosja może jednak wpaść w nieodwracalną zależność od Chin. Decydując się na inwazję na Ukrainę, Putin chciał przede wszystkim zasilić ludność Rosji dziesiątkami milionów Słowian, co było dla niego ważniejsze od zdobyczy terytorialnych. W tle są obawy przed chińską dominacją."""

## QA Generator

In [ ]:
qa_generator_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are an expert assistant for generating high-quality question–answer pairs from text.

Rules:
- Generate 12-15 question–answer pairs.
- Questions and answers MUST cover ONLY the most important aspects of the text.
- Questions and answers MUST be in Polish.
- Each question must be specific, concrete, and directly refer to explicit concepts, mechanisms, facts, events, or arguments present in the text.
- Do NOT ask generic or meta-level questions about the text itself.
- Forbidden example of a question: "Jakie jest główne zagadnienie omawiane w tekście?"
- Questions must test understanding of the content, not summarize it at a high level.
- Avoid trivial, vague, or purely definitional questions.
- Answers must be concise, precise, and based only on the provided text.

Output format:
- Return ONLY valid JSON.
- The JSON must be a list of objects.
- Each object must have exactly two fields: "question" and "answer".
- Do NOT include markdown, comments, or any text outside the JSON.

Example output:
[
  {{
    "question": "Jaką rolę pełni mechanizm X w opisywanym procesie?",
    "answer": "Mechanizm X odpowiada za ..."
  }},
  {{
    "question": "Jakie konsekwencje według tekstu wynikają z zastosowania rozwiązania Y?",
    "answer": "Zastosowanie rozwiązania Y prowadzi do ..."
  }}
]"""),
    ("user", """Generate question–answer pairs from the following text.
Text:\n{text}""")
])

llm_chain = qa_generator_prompt | llm_text
output = llm_chain.invoke({"text": text}).content
print(repr(output))

In [ ]:
qa_pairs = json.loads(output)
for i, pair in enumerate(qa_pairs, start=1):
    print(f"{i}. Question: {pair['question']}")
    print(f"   Answer: {pair['answer']}\n")

In [ ]:
allowed_nodes = [
    "Person",
    "Country",
    "Organization",
    "Alliance",
    "Conflict",
    "Event",
    "Ideology",
    "Technology",
    "EconomicSector",
    "Culture",
    "HistoricalEvent",
    "Policy"
]
allowed_relationships = [
    "PERCEIVES_AS",
    "ASSUMES_POSSIBILITY_OF",
    "STRENGTHENS_COOPERATION",
    "SUPPLIES_TECHNOLOGY",
    "DRAWS_HISTORICAL_LESSONS",
    "DENIES_NECESSITY_OF",
    "DIVIDES",
    "FEARS_DEMOCRATIZATION",
    "BECOMES_DEPENDENT_ON",
    "AIMS_FOR_DEMOGRAPHIC_REINFORCEMENT",
    "OPINION",
    "DECIDED_TO"
]

pairs = [f"Question: {pair['question']}\nAnswer: {pair['answer']}" for pair in qa_pairs]
node_properties=True
relationship_properties=True
llm_transformer = LLMGraphTransformer(
    llm=llm_text,
    allowed_nodes=allowed_nodes,
    allowed_relationships=allowed_relationships,
    node_properties=node_properties,
    relationship_properties=relationship_properties,
    strict_mode=False,
    additional_instructions="Extract ONLY 3-4 most important nodes and ONLY 1-2 most important relationships between them. Write everything in Polish.",
)

graph_documents = llm_transformer.convert_to_graph_documents([Document(t) for t in pairs])
print([doc for doc in graph_documents])

In [ ]:
for doc_idx, graph_doc in enumerate(graph_documents, start=1):
    print(f"\n=== DOCUMENT {doc_idx} ===")
    print("NODES:")
    for i, node in enumerate(graph_doc.nodes, start=1):
        print(f"  {i:02d}. {node.id} ({node.type})")

    print("RELATIONSHIPS:")
    for i, rel in enumerate(graph_doc.relationships, start=1):
        source = rel.source.id
        target = rel.target.id
        rel_type = rel.type
        print(f"  {i:02d}. {source} ──[{rel_type}]──▶ {target}")

## All pairs as a single string

In [ ]:
all_pairs = [f"Question {i}: {pair['question']}\nAnswer {i}: {pair['answer']}" for i, pair in enumerate(qa_pairs)]
single_sting = "\n" + "-" * 40 + "\n"
single_sting = "\n\n".join(all_pairs)

allowed_nodes = [
    "Person",
    "Country",
    "Organization",
    "Alliance",
    "Conflict",
    "Event",
    "Ideology",
    "Technology",
    "EconomicSector",
    "Culture",
    "HistoricalEvent",
    "Policy"
]
allowed_relationships = [
    "PERCEIVES_AS",
    "ASSUMES_POSSIBILITY_OF",
    "STRENGTHENS_COOPERATION",
    "SUPPLIES_TECHNOLOGY",
    "DRAWS_HISTORICAL_LESSONS",
    "DENIES_NECESSITY_OF",
    "DIVIDES",
    "FEARS_DEMOCRATIZATION",
    "BECOMES_DEPENDENT_ON",
    "AIMS_FOR_DEMOGRAPHIC_REINFORCEMENT",
    "OPINION",
    "DECIDED_TO"
]
node_properties=True
relationship_properties=True
llm_transformer = LLMGraphTransformer(
    llm=llm_text,
    allowed_nodes=allowed_nodes,
    allowed_relationships=allowed_relationships,
    node_properties=node_properties,
    relationship_properties=relationship_properties,
    strict_mode=False,
    additional_instructions="""Extract 7-8 most important nodes (real-world entities: countries, persons, organizations, events, concepts) and 6-7 most important factual relationships between them.

Critical rules (strict):
- DO NOT extract rhetorical, or trivial relationships, such as: believes, thinks, claims, says, suggests, fears.
- Relationships must describe objective actions, dependencies, or states with any significance.
- Each relationship must have a clear standalone meaning outside the sentence context (a single “X -> Y” must convey real information on its own).
- Write everything in Polish.

Importance filter:
- If a relationship does not change the understanding of the situation, omit it.
- Remove duplicates and trivial relationships.""",
)

graph_documents = llm_transformer.convert_to_graph_documents([Document(single_sting)])

In [ ]:
for doc_idx, graph_doc in enumerate(graph_documents, start=1):
    print(f"\n=== DOCUMENT {doc_idx} ===")
    print("NODES:")
    for i, node in enumerate(graph_doc.nodes, start=1):
        print(f"  {i:02d}. {node.id} ({node.type})")

    print("RELATIONSHIPS:")
    for i, rel in enumerate(graph_doc.relationships, start=1):
        source = rel.source.id
        target = rel.target.id
        rel_type = rel.type
        print(f"  {i:02d}. {source} ──[{rel_type}]──▶ {target}")

## Not restricted

In [ ]:
all_pairs = [f"Question {i}: {pair['question']}\nAnswer {i}: {pair['answer']}" for i, pair in enumerate(qa_pairs)]
single_sting = "\n" + "-" * 40 + "\n"
single_sting = "\n\n".join(all_pairs)

allowed_nodes = [
    "Person",
    "Country",
    "Organization",
    "Alliance",
    "Conflict",
    "Concept"
    "Event",
    "Ideology",
    "Technology",
    "EconomicSector",
    "Culture",
    "HistoricalEvent",
    "Policy"
]
llm_transformer = LLMGraphTransformer(
    llm=llm_text,
    allowed_nodes=allowed_nodes,
    node_properties=True,
    relationship_properties=True,
    strict_mode=False,
    additional_instructions="""# Role
You are an expert Data Analyst specializing in Knowledge Graph extraction. Your goal is to extract a highly precise, information-dense graph from the provided question-answer pairs.

# Task
- Extract the most important Nodes (Entities, Concepts, Events) and Relationships between them.
- Language of output: POLISH.

# Critical Rules for Relationships (Strict Enforcement)
- A triplet (Node A -> Relation -> Node B) must be a complete, factual sentence on its own, understandable without reading the original text.
- The relationship label must capture the *specific nuance* of the interaction. It should be a short phrase.
   - BAD: Trump ──[UWAŻA]──▶ Konflikt_w_Ukrainie  (Reason: Poor information provided from this triple. The verb is too vague and fails to describe the nature of the stance or the intended action.)
   - BAD: Putin ──[ZMNIEJSZA]──▶ Ryzyko  (Reason: Lack of causality or method. It doesn't explain how or why the risk is being reduced, making the data point nearly useless for analysis.)
   - BAD: Putin I Xi ──[UWAŻAJĄ,_ŻE]──▶ Rozpad Związku Radzieckiego  (Reason: Compound subjects violate graph atomicity. Putin and Xi should be separate nodes to allow for distinct relationships and to map their individual influence accurately.)
- Extract only objective actions, specific plans, defined causes, or verified states. Ignore rhetorical fluff.

# Critical Rules for Nodes
- OMIT META-NODES. Do not create nodes for 'The Author', 'The Text', or 'The Article'. If the author refers to themselves in the first person ('I believe...', 'My analysis...'), extract the core fact itself as the relationship between the primary entities involved.
- Resolve ambiguities. If text says "He said...", extract the actual name (e.g., "Trump"), not "He".
- Keep nodes precise but atomic (e.g., "Inwazja na Ukrainę", not "Straszna inwazja na Ukrainę o której pisał autor").""",
)

graph_documents = llm_transformer.convert_to_graph_documents([Document(single_sting)])

In [ ]:
for doc_idx, graph_doc in enumerate(graph_documents, start=1):
    print(f"\n=== DOCUMENT {doc_idx} ===")
    print("NODES:")
    for i, node in enumerate(graph_doc.nodes, start=1):
        print(f"  {i:02d}. {node.id} ({node.type})")

    print("RELATIONSHIPS:")
    for i, rel in enumerate(graph_doc.relationships, start=1):
        source = rel.source.id
        target = rel.target.id
        rel_type = rel.type
        print(f"  {i:02d}. {source} ──[{rel_type}]──▶ {target}")